# 03 Train Petrobras Models

This notebook trains the Petrobras models from the processed CSV and stores all outputs under a run-specific artifact directory.

In [ ]:
import os
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

def find_repo_root() -> Path:
    env_repo = os.getenv('PIPELINE_NOTEBOOK_REPO_ROOT')
    if env_repo:
        return Path(env_repo).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'scripts' / 'train_petrobras_models.py').exists():
            return candidate
    return Path('/content/workspace/pipeline-leak-detection')

REPO_ROOT = find_repo_root()
default_storage = Path('/content/drive/MyDrive/pipeline-leak-detection') if Path('/content').exists() else REPO_ROOT / 'artifacts' / 'notebook_runs'
STORAGE_ROOT = Path(os.getenv('PIPELINE_NOTEBOOK_STORAGE_ROOT', str(default_storage))).expanduser().resolve()
RUN_LABEL = os.getenv('PIPELINE_NOTEBOOK_RUN_LABEL', 'petrobras_full_run_01')

RUN_ROOT = STORAGE_ROOT / 'artifacts' / 'petrobras' / RUN_LABEL
PROCESSED_CSV = RUN_ROOT / 'data' / 'petrobras_3w_scada.csv'
MODELS_DIR = RUN_ROOT / 'models'
TRAINING_SUMMARY = MODELS_DIR / 'petrobras_training_summary.json'
METRICS_CSV = MODELS_DIR / 'petrobras_model_metrics.csv'

MODELS_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'train_petrobras_models.py'),
        '--input',
        str(PROCESSED_CSV),
        '--output',
        str(MODELS_DIR),
        '--summary-json',
        str(TRAINING_SUMMARY),
        '--metrics-csv',
        str(METRICS_CSV),
        '--run-label',
        RUN_LABEL,
    ],
    cwd=REPO_ROOT,
    check=True,
)


In [ ]:
with open(TRAINING_SUMMARY, 'r', encoding='utf-8') as f:
    training_summary = json.load(f)

training_summary['best_model']


In [ ]:
pd.read_csv(METRICS_CSV).sort_values(['roc_auc', 'f1'], ascending=[False, False])
